# Expanded A/B logit validation

400 MMLU source questions, 20 subjects, two option orders and three opinion conditions = 2,400 prompts.
This is a transformed binary subset, not the original four-choice MMLU score. Candidate likelihood is not calibrated factual confidence.

Select a GPU runtime. Run the setup and pre-report cells first, read/download the PDF, then run the separate inference cell. All results are checkpointed after each prompt. No model training or hidden-state experiment is included.

In [ ]:
from pathlib import Path
import sys, subprocess, json, base64, hashlib, html, mimetypes
from datetime import datetime, timezone
from IPython.display import HTML, display
import torch

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
REPO_DIR = Path("/content/latent-truth-xai")
REPO_URL = "https://github.com/seungjoolee24/XAI_research"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Using existing checkout. Pull the reviewed version explicitly if needed.")
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pytest"], cwd=REPO_DIR, check=True)
CONFIG_PATH = REPO_DIR / "configs/expanded.yaml"
RUN_DIR = REPO_DIR / "outputs" / ("expanded_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
PRE_REPORT_DELIVERED = False

def offer_file(path, label):
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    href = "data:" + mime + ";base64," + base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<p><a href="{href}" download="{html.escape(path.name, quote=True)}">{html.escape(label)}</a></p>'))
    print(path.name, "bytes:", path.stat().st_size, "SHA256:", hashlib.sha256(path.read_bytes()).hexdigest())
print("Output directory:", RUN_DIR)

## Before inference

Generate and deliver the frozen protocol PDF. Changes to config, dataset, source code, manifest or this PDF will block execution and require a new protocol.

In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / "scripts/run_validation.py"),
                "--config", str(CONFIG_PATH), "--run-dir", str(RUN_DIR), "--prepare"],
               cwd=REPO_DIR, check=True)
offer_file(RUN_DIR / "validation_pre_experiment.pdf", "Download pre-experiment PDF")
PRE_REPORT_DELIVERED = True
print("Pre-experiment PDF supplied. The next cell starts research inference.")

## GPU experiment

Execute only after the pre-experiment PDF above has been supplied. For a genuine interrupted run, keep RUN_DIR and use RESUME=True with the same unchanged protocol. Repeated execution without resume never overwrites a checkpoint.

In [ ]:
assert PRE_REPORT_DELIVERED, "Run and deliver the pre-report cell first."
RESUME = False
command = [sys.executable, str(REPO_DIR / "scripts/run_validation.py"),
           "--config", str(CONFIG_PATH), "--run-dir", str(RUN_DIR),
           "--execute", "--pre-report-delivered"]
if RESUME:
    command.append("--resume")
try:
    subprocess.run(command, cwd=REPO_DIR, check=True)
finally:
    post = RUN_DIR / "validation_post_experiment.pdf"
    if post.is_file():
        offer_file(post, "Download actual post-experiment PDF")
    else:
        print("No post PDF found; retain the traceback and all checkpoints.")
    # Offer the complete archive immediately, before a temporary runtime can expire.
    import shutil
    archive = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    offer_file(archive, "Download complete expanded experiment archive")

## Actual results and export

Cluster bootstrap resamples source questions, keeping both option orders together. Read strict formatting and semantic supplemental results separately. Subject labels describe source curricula, not measured item difficulty.

In [ ]:
import pandas as pd
import shutil
status_file = RUN_DIR / "completion.json"
if not status_file.is_file():
    status_file = RUN_DIR / "failure.json"
print(json.loads(status_file.read_text()) if status_file.is_file() else "No final status; archive may contain partial checkpoints.")
for filename in ("validation_by_condition.csv", "validation_order_consistency.csv"):
    if (RUN_DIR / filename).is_file():
        display(pd.read_csv(RUN_DIR / filename))
archive = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
offer_file(archive, "Download complete expanded experiment archive")